# Philadelphia — OPA methodology with LYCD land (revenue-neutral reassessment)

**What this models.** Philadelphia's real estate tax exactly as it is structured today — one flat
combined City + School District rate on assessed land plus building, with the Homestead Exemption,
the 10-year construction abatement and institutional exemptions all left in place — but with the
**land component re-valued by LYCD**. The rate is rolled back so the total levy is unchanged, the
anti-windfall convention in `lvt.reassessment`. No split rate and no LVT: this is the reassessment
question that precedes any rate change — *who wins and who loses purely from correcting the land
values?*

**How it differs from `model_lycd.ipynb`.** That notebook changes two things at once (the land
surface and the rate structure). This one changes only the land surface, and Step 8 then stacks a
4:1 split-rate on top of the reassessed base and separates the two effects with
`decompose_reassessment_and_lvt`.

**Two shared pieces of `lvt.philadelphia` carry the modeling choices**, so this notebook cannot drift
from the LVT notebooks by construction:
- `compute_lycd_land_values` — the GMA-hierarchical LYCD construction (lot-area chain, zone medians,
  KNN fallback, market-value cap). Step 2 checks its land base against the tracked LYCD export.
- `carry_forward_exemptions` — how each parcel's existing relief is re-applied when only land moves
  (homestead re-applied at `min(cap, value)`, abatement and partial relief carried in dollars,
  full institutional exemption kept). Step 5 checks it by reconstructing OPA's own taxable base.

In [1]:
import os
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
)
from lvt.reassessment import (
    model_revenue_neutral_reassessment,
    decompose_reassessment_and_lvt,
    save_reassessment_export,
    reassessment_equity,
)
from lvt.viz import create_city_report, reassessment_equity_chart
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups
from lvt.philadelphia import (
    tax_year_params, parcel_cache_path, split_zero_building_parcels,
    compute_lycd_land_values, carry_forward_exemptions,
)

CITY_NAME = 'philadelphia'
STATE_FIPS = '42'
COUNTY_FIPS = '101'
LAND_IMPROVEMENT_RATIO = 4.0   # used only by the Step 8 decomposition

# --- Tax year (see lvt/philadelphia.py; do not hardcode a millage here) ---
TAX_YEAR = int(os.environ.get('LVT_TAX_YEAR', 2026))   # override: LVT_TAX_YEAR=2027
TY = tax_year_params(TAX_YEAR)
MILLAGE = TY.combined_mills
PARCEL_PATH = parcel_cache_path(TAX_YEAR)
MODEL_TYPE = f'reassessment:lycd_land_ty{TAX_YEAR}'
EXPORT_SUFFIX = f'_lycd_reassessment_ty{TAX_YEAR}'
EXPORT_CITY = f'{CITY_NAME}{EXPORT_SUFFIX}'
REPORT_DIR = Path('../../analysis/reports') / EXPORT_CITY
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(TY.describe())
print(f'source: {TY.source}')

DATA_DIR = Path('data')
GMA_PATH = DATA_DIR / 'parcel_gma_assignment.parquet'
PIN_AREA_PATH = DATA_DIR / 'parcel_areas_by_pin_current.parquet'
# The tracked LYCD export for the same tax year; Step 2 uses it as a drift check.
LYCD_EXPORT_PATH = Path(f'../../analysis/data/{CITY_NAME}_lycd_ty{TAX_YEAR}.csv')

TY2026: 0.6159% city + 0.7839% school = 1.3998% (13.998 mills) | city target $891,102,000 (projection) | homestead $100,000
source: Rate: City of Philadelphia Quarterly City Managers Report, period ending 2026-03-31, Summary Table R-1 ('FY 2026 Tax Rate: .6159% City plus .7839% School District Total 1.3998%'). Target: same report, Table R-2, Real Property 'Current' full-year Current Projection ($891,102k). This is a Q3 projection, not a closed-out actual.


C:\Users\druss\miniconda3\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Step 1: Load parcel data

In [2]:
if not PARCEL_PATH.exists():
    raise FileNotFoundError(
        f'{PARCEL_PATH} not found. Build it with:\n'
        f'    python scripts/build_philadelphia_parcel_cache.py --year {TAX_YEAR}\n'
        'The cache is keyed by tax year on purpose — opa_properties_public always carries '
        'the latest assessment year, so an unsuffixed cache makes it easy to model one '
        "year's taxable values against another year's expectations with no visible symptom."
    )
gdf = gpd.read_parquet(PARCEL_PATH)
_required = {'parcel_number', 'taxable_land', 'taxable_building', 'market_value',
             'exempt_land', 'exempt_building', 'pin', 'category_code', 'total_area',
             'homestead_exemption'}
_missing = _required - set(gdf.columns)
if _missing:
    raise ValueError(
        f'{PARCEL_PATH} is missing columns {sorted(_missing)} — rebuild it with '
        f'scripts/build_philadelphia_parcel_cache.py --year {TAX_YEAR} --force'
    )
gdf['parcel_number'] = gdf['parcel_number'].astype(str).str.zfill(9)
for _c in ['taxable_land', 'taxable_building', 'market_value', 'exempt_land', 'exempt_building']:
    gdf[_c] = pd.to_numeric(gdf[_c], errors='coerce').fillna(0.0)

# OPA's gross (pre-exemption) components. market_value = gross land + gross building on all
# but a handful of records; the reform replaces gross land and leaves gross building alone.
gdf['opa_gross_land'] = (gdf['taxable_land'] + gdf['exempt_land']).clip(lower=0)
gdf['opa_gross_building'] = (gdf['taxable_building'] + gdf['exempt_building']).clip(lower=0)
_id_gap = (gdf['opa_gross_land'] + gdf['opa_gross_building'] - gdf['market_value']).abs() > 1
print(f'Loaded {len(gdf):,} parcels for TY{TAX_YEAR}')
print(f'  taxable base:      ${(gdf["taxable_land"].sum() + gdf["taxable_building"].sum())/1e9:.3f}B')
print(f'  gross land:        ${gdf["opa_gross_land"].sum()/1e9:.3f}B   gross building: ${gdf["opa_gross_building"].sum()/1e9:.3f}B')
print(f'  records where gross land + building != market_value: {int(_id_gap.sum()):,}')

Loaded 583,249 parcels for TY2026
  taxable base:      $152.997B
  gross land:        $53.778B   gross building: $179.326B
  records where gross land + building != market_value: 10


## Step 2: LYCD land values (shared construction)

`compute_lycd_land_values` is the same code path `model_lycd.ipynb` uses: one lot-area convention
(OPA `total_area`, then Mercator-corrected PIN polygon area, then KNN), GMA-hierarchical zone medians
of total $/sqft over improved parcels, 20% allocation for improved parcels and 100% for vacant ones,
KNN fallback outside GMA coverage, and the improved-only market-value cap. Its diagnostics are the
numbers the LVT notebooks print; see `docs/LYCD_LAND_MODEL_ROADMAP.md` for what the method is and
is not.

**Drift check.** The tracked LYCD export for this tax year records the same land base over taxable
parcels (`taxable_land_value` = LYCD land for every non-exempt parcel). Identical construction on
identical inputs must reproduce it to the dollar; anything else means the export is stale or the
shared function has diverged from the notebook.

In [3]:
for _p in (GMA_PATH, PIN_AREA_PATH):
    if not _p.exists():
        raise FileNotFoundError(f'{_p} not found — see model_lycd.ipynb Step 2/3 for how to build it.')
gma = pd.read_parquet(GMA_PATH)
pin_areas = pd.read_parquet(PIN_AREA_PATH)

lycd = compute_lycd_land_values(gdf, gma, pin_areas)
gdf = lycd.gdf
print(lycd.describe())

_taxable_now = (gdf['taxable_land'] + gdf['taxable_building']).clip(lower=0) > 0
_lycd_taxable_base = float(gdf.loc[_taxable_now, 'lycd_land_value'].sum())
print(f'\nLYCD land base (all parcels):     ${gdf["lycd_land_value"].sum()/1e9:.3f}B')
print(f'OPA gross land (all parcels):     ${gdf["opa_gross_land"].sum()/1e9:.3f}B')
print(f'LYCD / OPA gross land:            {gdf["lycd_land_value"].sum()/gdf["opa_gross_land"].sum():.2f}x')

if LYCD_EXPORT_PATH.exists():
    _exp = pd.read_csv(LYCD_EXPORT_PATH, usecols=['taxable_land_value'])
    _exp_base = float(_exp['taxable_land_value'].sum())
    _rel = abs(_lycd_taxable_base - _exp_base) / _exp_base
    print(f'\nDrift check vs {LYCD_EXPORT_PATH.name}: this run ${_lycd_taxable_base/1e9:.6f}B '
          f'| export ${_exp_base/1e9:.6f}B | rel diff {_rel:.2e}')
    assert _rel < 1e-6, (
        'The shared LYCD construction no longer reproduces the tracked LYCD export. Either the '
        'export is stale (re-run model_lycd.ipynb) or compute_lycd_land_values has drifted from '
        'that notebook. Do not proceed on a land base the LVT notebooks do not share.'
    )
else:
    print(f'\n(no {LYCD_EXPORT_PATH.name} to check against — run model_lycd.ipynb to enable the drift check)')

Lot area source: opa_total_area=550,807, knn=30,696, pin_dor=1,351, pin_override=395
  OPA records overridden by surveyed polygon: 395
  total lot area = 0.84x the city (expect <1.0)
GMA assignment: 527,364 matched (90.4%); L3=526,329, knn=55,885, L2=929, L1=106
Market-value cap (improved only): 21,727 parcels, $106.41B -> $82.37B (22.6% removed)

LYCD land base (all parcels):     $82.367B
OPA gross land (all parcels):     $53.778B
LYCD / OPA gross land:            1.53x



Drift check vs philadelphia_lycd_ty2026.csv: this run $62.543155B | export $62.543155B | rel diff 1.22e-16


## Step 3: Categorize parcels (same overrides as the LVT notebooks)

In [4]:
gdf['category_code'] = (
    pd.to_numeric(gdf['category_code'], errors='coerce')
    .astype('Int64')
    .astype(str)
)

CATEGORY_MAP = {
    '1':  'Single Family Residential',
    '2':  'Small Multi-Family (2-4 units)',
    '3':  'Mixed Use',
    '4':  'Commercial',
    '5':  'Industrial',
    '6':  'Vacant Land',
    '7':  'Other Commercial',
    '8':  'Other Residential',
    '9':  'Hotel',
    '10': 'Office / Commercial Condo',
    '11': 'Other',
    '12': 'Vacant Land',
    '13': 'Vacant Land',
    '14': 'Large Multi-Family (5+ units)',
    '15': 'Retail / General Commercial',
}
gdf['PROPERTY_CATEGORY'] = gdf['category_code'].map(CATEGORY_MAP).fillna('Other')

# Override 1: $0 improvement -> Vacant Land
gdf.loc[gdf['taxable_building'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

# Override 2: a $0 taxable building line has three different causes (abatement, homestead
# larger than the building line, genuinely $0 improvement). Split on the year's statutory cap.
GENUINE_VACANT_CODES = {'6', '12', '13'}
_zb = split_zero_building_parcels(
    gdf, gdf['PROPERTY_CATEGORY'], TY.homestead_exemption, CATEGORY_MAP,
    genuine_vacant_codes=tuple(GENUINE_VACANT_CODES),
)
gdf['PROPERTY_CATEGORY'] = _zb.category
abated_mask = _zb.abated
print(_zb.describe())

# Override 3: OPA-vacant with nonzero building value
improved_vacant_mask = (
    gdf['category_code'].isin(GENUINE_VACANT_CODES) &
    (gdf['taxable_building'] > 0)
)
gdf.loc[improved_vacant_mask, 'PROPERTY_CATEGORY'] = 'Improved Vacant Land'

gdf['taxable_total'] = (gdf['taxable_land'] + gdf['taxable_building']).clip(lower=0)
gdf['full_exmp'] = (gdf['taxable_total'] <= 0).astype(int)

# Override 4: fully exempt parcels
EXEMPT_CATEGORY_MAP = {k: v + ' — Exempt' for k, v in CATEGORY_MAP.items()}
exempt_mask = gdf['full_exmp'] == 1
gdf.loc[exempt_mask, 'PROPERTY_CATEGORY'] = (
    gdf.loc[exempt_mask, 'category_code']
    .map(EXEMPT_CATEGORY_MAP)
    .fillna('Other — Exempt')
)

print(f'Total parcels: {len(gdf):,}')
print(f'Fully exempt today: {gdf["full_exmp"].sum():,}  |  '
      f'Abated: {abated_mask.sum():,}  |  '
      f'Improved vacant: {improved_vacant_mask.sum():,}  |  '
      f'Taxable today: {(gdf["full_exmp"] == 0).sum():,}')

zero-building line: 14,287 abated | 13,995 homestead-zeroed (96.3% confirmed by OPA's homestead flag) | 1,119 genuinely $0 improvement
Total parcels: 583,249
Fully exempt today: 36,932  |  Abated: 14,287  |  Improved vacant: 880  |  Taxable today: 546,317


## Step 4: Current tax (OPA taxable values — the revenue baseline)

In [5]:
gdf['millage_rate'] = MILLAGE

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='taxable_total',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

city_revenue = gdf['taxable_total'].mul(TY.city_mills / 1000).sum()

print(f'Modeled combined levy (city + school):  ${current_revenue:,.0f}')
print(f'Implied city-only portion ({TY.city_rate_pct}%):   ${city_revenue:,.0f}')

if TY.city_revenue_target is None:
    print(f'\nNO REVENUE VALIDATION for TY{TAX_YEAR}.')
    print(f'  {TY.source}')
else:
    gap_pct = (city_revenue / TY.city_revenue_target - 1) * 100
    print(f'City-only target ({TY.target_kind}):            ${TY.city_revenue_target:,}')
    print(f'City portion gap: {gap_pct:+.2f}%  (expected: a few % over, from delinquency)')
    assert abs(gap_pct) < 10.0, (
        f'City gap {gap_pct:.2f}% exceeds 10% for TY{TAX_YEAR}. Check that the assessment '
        f'year, the City rate ({TY.city_rate_pct}%) and the revenue target all refer to the '
        'same billing year — see lvt/philadelphia.py.'
    )

Modeled combined levy (city + school):  $2,141,653,043
Implied city-only portion (0.6159%):   $942,308,979
City-only target (projection):            $891,102,000
City portion gap: +5.75%  (expected: a few % over, from delinquency)


## Step 5: The reassessed base — LYCD land, OPA building, every exemption carried forward

"Same methodology, land valued differently" means every relief that exists today still exists
tomorrow, applied the way OPA applies it. `carry_forward_exemptions` does that in three parts:

| Relief | How OPA records it | How it carries forward |
|---|---|---|
| Homestead Exemption | flat statutory amount (`homestead_exemption` column), building line first, remainder to land | re-applied as `min(cap, new total)` — a homestead whose LYCD land lifts it above the cap becomes taxable again |
| 10-year abatement, partial institutional relief | dollars in `exempt_building` / `exempt_land` beyond the homestead | carried in dollars, building first — the building value is unchanged by a land reform, so an abatement's dollars are exactly right |
| Full institutional exemption | 100% of value, not an amount | stays fully exempt |

**Guard.** Feeding OPA's *own* gross land back through the same rule must reproduce OPA's taxable
total per parcel. If the rule mis-described this vintage's exemptions, that reconstruction rate is
the number that would move; the function enforces a floor and reports it. The homestead flag is
current-vintage (the cache builder documents this), so parcels flagged today with no exemption in
this year's assessment are counted and not re-exempted.

Note the LVT notebooks do *not* re-apply the homestead to LYCD land (their `model_building` is
post-exemption and their land is gross), so their SFR results embed a slightly larger land base
than this notebook's. That is a known simplification there, not a discrepancy here.

In [6]:
cf = carry_forward_exemptions(
    gdf, new_land_col='lycd_land_value', homestead_cap=TY.homestead_exemption,
)
gdf['reform_taxable_land'] = cf.reform_taxable_land
gdf['reform_taxable_building'] = cf.reform_taxable_building
gdf['reform_taxable_total'] = cf.reform_taxable_total
gdf['institutional_exempt'] = cf.institutional_exempt.astype(int)
gdf['homestead_active'] = cf.homestead_active.astype(int)
gdf['reentered_base'] = ((gdf['full_exmp'] == 1) & (gdf['reform_taxable_total'] > 0)).astype(int)
print(cf.describe())

# A parcel that pays under the reform cannot stay labelled "Exempt": relabel the re-entering
# cohort so the category tables say what happened to it.
_re = gdf['reentered_base'] == 1
gdf.loc[_re, 'PROPERTY_CATEGORY'] = (
    gdf.loc[_re, 'PROPERTY_CATEGORY'].str.replace(' — Exempt', '', regex=False) + ' — Re-entering (homestead-wiped today)'
)

_d = cf.diagnostics
print(f"\nReconstruction mismatch: ${_d['reconstruction_mismatch_dollars']/1e6:,.1f}M of taxable value "
      f"on {(1-_d['reconstruction_match_rate'])*len(gdf):,.0f} parcels "
      f"({_d['reconstruction_mismatch_dollars']/gdf['taxable_total'].sum():.3%} of the OPA base)")

print('\nBase comparison ($B):')
_base = pd.DataFrame({
    'OPA today': [gdf['taxable_land'].clip(lower=0).sum(), gdf['taxable_building'].clip(lower=0).sum(), gdf['taxable_total'].sum()],
    'LYCD land, exemptions carried': [gdf['reform_taxable_land'].sum(), gdf['reform_taxable_building'].sum(), gdf['reform_taxable_total'].sum()],
}, index=['taxable land', 'taxable building', 'taxable total']) / 1e9
_base['ratio'] = _base.iloc[:, 1] / _base.iloc[:, 0]
print(_base.round(3).to_string())

print(f'\nParcels exempt today that become taxable: {int(_re.sum()):,} '
      f'(taxable value ${gdf.loc[_re, "reform_taxable_total"].sum()/1e9:.3f}B)')
print(gdf.loc[_re, 'PROPERTY_CATEGORY'].value_counts().head(6).to_string())

exemption carry-forward: 233,326 homestead parcels re-exempted at min(cap, value) | 60,859 carry non-homestead dollars (abatement / partial) | 23,644 stay fully exempt | 5,582 homestead-wiped parcels re-enter the base | flag-without-exemption (vintage mismatch): 8,348 | OPA-base reconstruction match: 99.50%

Reconstruction mismatch: $356.2M of taxable value on 2,923 parcels (0.233% of the OPA base)

Base comparison ($B):
                  OPA today  LYCD land, exemptions carried  ratio
taxable land         43.000                         62.066  1.443
taxable building    109.997                        109.969  1.000
taxable total       152.997                        172.034  1.124

Parcels exempt today that become taxable: 5,582 (taxable value $0.081B)
PROPERTY_CATEGORY
Single Family Residential — Re-entering (homestead-wiped today)         5520
Mixed Use — Re-entering (homestead-wiped today)                           34
Small Multi-Family (2-4 units) — Re-entering (homestead-wiped toda

## Step 6: Revenue-neutral flat-rate reassessment

One flat rate on the reassessed base, rolled back so the levy equals today's. Parcels that stay
fully exempt are excluded from the denominator; parcels re-entering the base are in it with a
current tax of zero. A parcel pays less if and only if its taxable-value ratio (reassessed / today)
is below the citywide ratio.

In [7]:
flat_millage, new_revenue, gdf = model_revenue_neutral_reassessment(
    gdf,
    new_land_col='reform_taxable_land',
    new_improvement_col='reform_taxable_building',
    current_revenue=current_revenue,
    old_value_col='taxable_total',
    exemption_flag_col='institutional_exempt',
    compute_current_tax=False,
    verbose=True,
)

# Re-entering parcels have current_tax = 0; the function writes 0% there, which would read as
# "no change". Match the export convention (null where current_tax == 0).
gdf.loc[gdf['current_tax'] <= 0, 'tax_change_pct'] = np.nan

_cw_ratio = gdf.loc[gdf['institutional_exempt'] == 0, 'reform_taxable_total'].sum() / \
            gdf.loc[gdf['institutional_exempt'] == 0, 'taxable_total'].sum()
print(f'\nCurrent millage:      {MILLAGE:.4f} mills')
print(f'Rolled-back millage:  {flat_millage:.4f} mills  ({flat_millage/MILLAGE-1:+.1%})')
print(f'Citywide taxable-value ratio (reassessed / today): {_cw_ratio:.4f}  '
      '-- a parcel wins iff its own ratio is below this')

category_summary = calculate_category_tax_summary(
    df=gdf, category_col='PROPERTY_CATEGORY', current_tax_col='current_tax', new_tax_col='new_tax',
)
print_category_tax_summary(
    category_summary,
    title=f'Philadelphia TY{TAX_YEAR} — flat-rate reassessment, LYCD land, exemptions carried forward',
)

Revenue-neutral reassessment (single district)
Rolled-back millage: 12.4490 per $1,000
New revenue: $2,141,653,042.83   Target: $2,141,653,042.83
Revenue difference: $0.00 (0.0000%)

Current millage:      13.9980 mills
Rolled-back millage:  12.4490 mills  (-11.1%)
Citywide taxable-value ratio (reassessed / today): 1.1244  -- a parcel wins iff its own ratio is below this



Philadelphia TY2026 — flat-rate reassessment, LYCD land, exemptions carried forward
                                                            Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
                                           Single Family Residential 430570    $-45,607,459       -3.7%      $-106        $-194   -2.9%     -10.8%            11.7%            55.5%
                                      Small Multi-Family (2-4 units)  38685    $-32,850,934      -13.0%      $-849        $-485  -10.4%     -12.6%             5.1%            65.0%
                                                         Vacant Land  30557    $136,116,567      260.4%     $4,455       $1,042  564.1%     253.1%            90.0%             8.0%
                                  Single Family Residential — Exempt  14558              $0        0.0%         $0           $0    0.0%       0.0%             0.0%             0.0%
          

## Step 7: Where the shift comes from

The reassessment moves money from parcels whose land LYCD values *below* OPA's share of the city
total to parcels it values above it. Three views: the land re-valuation itself (LYCD versus OPA gross
land, by category), the bill outcome by the GMA level the land value came from, and by the lot-area
source.

Read the `knn` rows of the last two tables with care. Parcels with no OPA lot area and no PIN
polygon (condominium units above all: OPA records the unit, DOR records the building's lot) receive a
neighbour's *whole-lot* area, so LYCD hands each unit a full lot's worth of land; the market-value cap
then pins their land at 100% of unit value. Their taxable ratio is therefore a cap artifact, not a
land-value finding. It is the KNN/cap defect in `docs/LYCD_LAND_MODEL_ROADMAP.md`, and its dollar
contribution is the `net $ shift` on the `knn` area-source row below.

In [8]:
_tx = gdf[(gdf['institutional_exempt'] == 0) & (gdf['current_tax'] > 0)].copy()
_tx['land_ratio'] = np.where(_tx['opa_gross_land'] > 0, _tx['lycd_land_value'] / _tx['opa_gross_land'], np.nan)
_tx['value_ratio'] = np.where(_tx['taxable_total'] > 0, _tx['reform_taxable_total'] / _tx['taxable_total'], np.nan)
_tx['wins'] = _tx['tax_change'] < 0

def _summ(g):
    return pd.Series({
        'n': len(g),
        'median LYCD/OPA land': g['land_ratio'].median(),
        'median taxable ratio': g['value_ratio'].median(),
        'winners %': 100 * g['wins'].mean(),
        'median $ change': g['tax_change'].median(),
        'median % change': g['tax_change_pct'].median(),
        'net $ shift (M)': g['tax_change'].sum() / 1e6,
    })

_cols = ['land_ratio', 'value_ratio', 'wins', 'tax_change', 'tax_change_pct']
print('By property category (taxable parcels with a current bill):')
print(_tx.groupby('PROPERTY_CATEGORY')[_cols].apply(_summ).sort_values('n', ascending=False).round(2).to_string())
print('\nBy GMA level the land value came from:')
print(_tx.groupby('gma_level')[_cols].apply(_summ).round(2).to_string())
print('\nBy lot-area source:')
print(_tx.groupby('area_source')[_cols].apply(_summ).round(2).to_string())

# Distribution of the per-parcel taxable-value ratio (the thing the flat rate acts on)
fig, ax = plt.subplots(figsize=(8, 4.5))
_r = _tx['value_ratio'].clip(0, 4)
ax.hist(_r, bins=120, color='#4C78A8')
ax.axvline(_cw_ratio, color='#E45756', lw=1.5, label=f'citywide ratio {_cw_ratio:.3f} (breakeven)')
ax.set_xlabel('reassessed taxable value / current taxable value (clipped at 4)')
ax.set_ylabel('parcels')
ax.set_title('Per-parcel reassessment ratio, LYCD land with exemptions carried forward')
ax.legend()
fig.tight_layout()
fig.savefig(REPORT_DIR / 'reassessment_ratio_distribution.png', dpi=150)
plt.close(fig)
print(f'\nSaved {REPORT_DIR / "reassessment_ratio_distribution.png"}')

By property category (taxable parcels with a current bill):
                                        n  median LYCD/OPA land  median taxable ratio  winners %  median $ change  median % change  net $ shift (M)
PROPERTY_CATEGORY                                                                                                                                  
Single Family Residential        430570.0                  1.01                  1.00      81.35          -194.25           -10.80           -45.61
Small Multi-Family (2-4 units)    38685.0                  0.93                  0.98      89.42          -485.38           -12.57           -32.85
Vacant Land                       30557.0                  3.96                  3.97       8.97          1041.89           253.07           136.12
Abated / Construction Exemption   14287.0                  0.66                  0.65      72.75          -646.18           -42.60           -12.09
Mixed Use                         13743.0           


Saved ..\..\analysis\reports\philadelphia_lycd_reassessment_ty2026\reassessment_ratio_distribution.png


## Step 8: Decomposition — reassessment versus split-rate

Stack a 4:1 split-rate on the *reassessed* base (revenue-neutral against the same levy) and separate
the two effects per parcel: `reassess_change` is the flat-rate reassessment above, `lvt_change` is the
rate-structure change on top of it, and the two sum to the total by construction. This is the same
`decompose_reassessment_and_lvt` the Reading notebooks use.

In [9]:
_solve = gdf[gdf['institutional_exempt'] == 0][
    ['reform_taxable_land', 'reform_taxable_building', 'current_tax', 'PROPERTY_CATEGORY']].copy()
lvt_land_millage, lvt_imp_millage, lvt_revenue, _solve = model_split_rate_tax(
    df=_solve,
    land_value_col='reform_taxable_land',
    improvement_value_col='reform_taxable_building',
    current_revenue=current_revenue,
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)
gdf['lvt_new_tax'] = 0.0
gdf.loc[_solve.index, 'lvt_new_tax'] = _solve['new_tax']
print(f'Split-rate on the reassessed base: land {lvt_land_millage:.4f} mills, '
      f'improvement {lvt_imp_millage:.4f} mills, revenue ${lvt_revenue:,.0f}')

gdf = decompose_reassessment_and_lvt(
    gdf, current_tax_col='current_tax', reassessed_tax_col='new_tax', final_tax_col='lvt_new_tax',
)
for _c in ['reassess_change_pct', 'lvt_change_pct', 'total_change_pct']:
    gdf.loc[gdf['current_tax'] <= 0, _c] = np.nan

_dec = gdf[(gdf['institutional_exempt'] == 0) & (gdf['current_tax'] > 0)]
_tbl = _dec.groupby('PROPERTY_CATEGORY').agg(
    n=('current_tax', 'size'),
    reassess_median_pct=('reassess_change_pct', 'median'),
    lvt_median_pct=('lvt_change_pct', 'median'),
    total_median_pct=('total_change_pct', 'median'),
    reassess_net_M=('reassess_change', lambda s: s.sum() / 1e6),
    lvt_net_M=('lvt_change', lambda s: s.sum() / 1e6),
).sort_values('n', ascending=False)
print('\nPer-category medians: reassessment effect vs split-rate effect (% of current bill), '
      'and net dollars moved ($M):')
print(_tbl.round(1).to_string())

Split-rate on the reassessed base: land 23.9136 mills, improvement 5.9784 mills, revenue $2,141,653,043



Per-category medians: reassessment effect vs split-rate effect (% of current bill), and net dollars moved ($M):
                                      n  reassess_median_pct  lvt_median_pct  total_median_pct  reassess_net_M  lvt_net_M
PROPERTY_CATEGORY                                                                                                        
Single Family Residential        430570                -10.8           -11.8             -21.0           -45.6      -98.7
Small Multi-Family (2-4 units)    38685                -12.6           -22.7             -32.3           -32.9      -46.6
Vacant Land                       30557                253.1            92.1             578.2           136.1      173.5
Abated / Construction Exemption   14287                -42.6            92.1              10.3           -12.1       22.7
Mixed Use                         13743                 -7.5           -18.2             -24.1            -4.0       -8.7
Commercial                       

## Step 9: Census join

In [10]:
import concurrent.futures

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f'Census join: {gdf["std_geoid"].notna().mean()*100:.1f}% matched')
        except concurrent.futures.TimeoutError:
            print('Census API timed out — skipping census join')
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f'Census join failed: {e}')
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

Census join: 100.0% matched


## Step 10: Who the reassessment helps and hurts

`reassessment_equity` stratifies winners and losers by block-group income quintile, by
racial-composition band, and by market-value decile (the vertical-equity view: does re-valuing land
cut cheap or expensive parcels harder?). Bootstrap intervals so thin strata read as a range.

In [11]:
_eq_mask = (gdf['institutional_exempt'] == 1) | (gdf['current_tax'] <= 0)
equity = reassessment_equity(
    gdf,
    value_col='market_value',
    n_boot=200, random_state=0,
    exclude_mask=_eq_mask,
)
show = ['n', 'pct_winners', 'pct_winners_lo', 'pct_winners_hi', 'median_change_pct']
for key in ['by_income_quintile', 'by_value_decile', 'by_minority_band']:
    tbl = equity[key]
    if not len(tbl):
        print(f'{key}: no data'); continue
    grp = tbl.columns[0]
    print(f'{key}  (winners %, 95% CI, median change %):')
    print(tbl[[grp] + [c for c in show if c in tbl.columns]].round(1).to_string(index=False))
    print()

for key, gcol, fname in [
    ('by_income_quintile', 'income_quintile', 'reassessment_equity_income.png'),
    ('by_value_decile',    'value_decile',    'reassessment_equity_value.png'),
    ('by_minority_band',   'minority_band',   'reassessment_equity_minority.png'),
]:
    tbl = equity[key]
    if len(tbl):
        reassessment_equity_chart(
            tbl, gcol,
            title=f"Philadelphia TY{TAX_YEAR} — LYCD-land reassessment by {gcol.replace('_', ' ')}\n"
                  "(blue = pay less, red = pay more)",
            save_path=str(REPORT_DIR / fname),
        )
        plt.close()
        print(f'Saved {REPORT_DIR / fname}')

by_income_quintile  (winners %, 95% CI, median change %):
income_quintile     n  pct_winners  pct_winners_lo  pct_winners_hi  median_change_pct
             Q1 93628         74.6            74.4            74.8              -10.3
             Q2 93806         80.0            79.7            80.2              -10.9
             Q3 94066         81.3            81.0            81.5              -10.8
             Q4 92889         75.6            75.3            75.9              -10.3
             Q5 93292         70.0            69.8            70.3               -9.5

by_value_decile  (winners %, 95% CI, median change %):
value_decile     n  pct_winners  pct_winners_lo  pct_winners_hi  median_change_pct
          D1 54645         37.9            37.5            38.3               55.6
          D2 54635         77.5            77.1            77.8              -10.2
          D3 54635         83.1            82.7            83.4              -11.0
          D4 54772         85.4       

## Step 11: Export and visualize

Standard 16-column export plus the reassessment columns (`reassessment_ratio`, the six
decomposition columns) and the per-parcel provenance this model adds (`lycd_land_value`,
`opa_gross_land`, `gma_level`, `homestead_active`, `institutional_exempt`, `reentered_base`).
`new_tax` / `tax_change` are the flat-rate reassessment; the split-rate result is only in the
decomposition columns.

In [12]:
EXTRA_COLS = [
    'reassessment_ratio',
    'reassess_change', 'reassess_change_pct', 'lvt_change', 'lvt_change_pct',
    'total_change', 'total_change_pct',
    'lycd_land_value', 'opa_gross_land', 'opa_gross_building', 'gma_level', 'area_source',
    'homestead_active', 'institutional_exempt', 'reentered_base',
]
out_df = save_reassessment_export(
    gdf,
    EXPORT_CITY,
    f'../../analysis/data/{EXPORT_CITY}.csv',
    model_type=MODEL_TYPE,
    land_millage=flat_millage,
    improvement_millage=flat_millage,
    extra_cols=EXTRA_COLS,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
    exempt_flag_col='institutional_exempt',
    parcel_id_col='parcel_number',
)
print(f'Exported {len(out_df):,} rows, {len(out_df.columns)} columns')

create_city_report(out_df, EXPORT_CITY, show=False)
print('Done.')

  [warn] philadelphia_lycd_reassessment_ty2026: non-standard property categories (will be preserved): ['Abated / Construction Exemption', 'Commercial — Exempt', 'Hotel — Exempt', 'Improved Vacant Land', 'Industrial — Exempt', 'Large Multi-Family (5+ units) — Exempt', 'Large Multi-Family (5+ units) — Re-entering (homestead-wiped today)', 'Mixed Use — Exempt', 'Mixed Use — Re-entering (homestead-wiped today)', 'Office / Commercial Condo — Exempt', 'Other Commercial — Exempt', 'Other Commercial — Re-entering (homestead-wiped today)', 'Other Residential — Exempt', 'Other Residential — Re-entering (homestead-wiped today)', 'Other — Exempt', 'Retail / General Commercial — Exempt', 'Single Family Residential — Exempt', 'Single Family Residential — Re-entering (homestead-wiped today)', 'Small Multi-Family (2-4 units) — Exempt', 'Small Multi-Family (2-4 units) — Re-entering (homestead-wiped today)', 'Vacant Land — Exempt', 'Vacant Land — Re-entering (homestead-wiped today)']


  ✓ philadelphia_lycd_reassessment_ty2026: 583,249 rows → ../../analysis/data/philadelphia_lycd_reassessment_ty2026.csv  [model: reassessment:lycd_land_ty2026]


Exported 583,249 rows, 32 columns


create_city_report: excluded 23,644 fully-exempt parcels (583,249 → 559,605 modeled).


Done.
